<a href="https://colab.research.google.com/github/Jaichiddharth/Digio-labs-assesment/blob/main/name_matcher.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [31]:
!mkdir -p digio-name-match/name_matcher
!mkdir -p digio-name-match/data
!mkdir -p digio-name-match/tests
%cd digio-name-match

/content/digio-name-match/digio-name-match/digio-name-match


In [32]:
%%writefile name_matcher/__init__.py
from .matcher import (
    score_names,
    decide,
    explain,
    MATCH_THRESHOLD,
    REVIEW_THRESHOLD,
)

__all__ = [
    "score_names",
    "decide",
    "explain",
    "MATCH_THRESHOLD",
    "REVIEW_THRESHOLD",
]

Writing name_matcher/__init__.py


In [33]:
%%writefile name_matcher/__main__.py
import json
import sys

from .matcher import explain


def main() -> None:
    if len(sys.argv) != 3:
        print("usage: python -m name_matcher 'Name 1' 'Name 2'")
        raise SystemExit(2)

    print(json.dumps(explain(sys.argv[1], sys.argv[2]), indent=2))


if __name__ == "__main__":
    main()

Writing name_matcher/__main__.py


In [34]:
%%writefile name_matcher/normalize.py
import re
import unicodedata
from typing import List

TITLES = {
    "mr", "mrs", "ms", "miss", "dr", "prof", "adv", "er",
    "ca", "cs", "shri", "sri", "shree", "smt", "shrimati", "late",
}

RELATION_RE = re.compile(
    r"\b(?:s/o|d/o|w/o|son of|daughter of|wife of)\b",
    re.IGNORECASE,
)

ALIASES = {
    "md": "mohammed",
    "mohd": "mohammed",
    "mohammad": "mohammed",
    "mohammed": "mohammed",
    "muhammad": "mohammed",
    "muhammed": "mohammed",
    "mohammd": "mohammed",
    "mohamed": "mohammed",
}


def _strip_accents(value: str) -> str:
    return "".join(
        ch
        for ch in unicodedata.normalize("NFKD", value)
        if not unicodedata.combining(ch)
    )


def normalize_tokens(raw: str) -> List[str]:
    text = _strip_accents(str(raw or ""))
    text = text.lower()
    text = text.replace("&", " and ")

    text = RELATION_RE.sub(" ", text)
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    tokens: List[str] = []
    for token in text.split():
        if token in TITLES:
            continue
        if token == "and":
            continue
        token = ALIASES.get(token, token)
        tokens.append(token)

    return tokens

Writing name_matcher/normalize.py


In [35]:
%%writefile name_matcher/matcher.py
from collections import Counter
from difflib import SequenceMatcher
from typing import Dict, List

from .normalize import normalize_tokens

MATCH_THRESHOLD = 0.72
REVIEW_THRESHOLD = 0.58


def _is_initial(token: str) -> bool:
    return len(token) == 1 and token.isalpha()


def _token_pair_score(a: str, b: str) -> float:
    if a == b:
        return 1.0
    if _is_initial(a) and b.startswith(a):
        return 0.90
    if _is_initial(b) and a.startswith(b):
        return 0.90
    if len(a) >= 3 and len(b) >= 3 and (a.startswith(b) or b.startswith(a)):
        return 0.85
    return SequenceMatcher(None, a, b).ratio()


def _exact_f1(a: List[str], b: List[str]) -> float:
    if not a or not b:
        return 0.0
    ca, cb = Counter(a), Counter(b)
    overlap = sum((ca & cb).values())
    if overlap == 0:
        return 0.0
    precision = overlap / len(b)
    recall = overlap / len(a)
    return 2 * precision * recall / (precision + recall)


def _fuzzy_f1(a: List[str], b: List[str]) -> float:
    if not a or not b:
        return 0.0
    used = set()
    matches = 0
    for ta in a:
        best_j = None
        best_s = 0.0
        for j, tb in enumerate(b):
            if j in used:
                continue
            s = _token_pair_score(ta, tb)
            if s > best_s:
                best_s = s
                best_j = j
        if best_j is not None and best_s >= 0.85:
            used.add(best_j)
            matches += 1
    if matches == 0:
        return 0.0
    precision = matches / len(b)
    recall = matches / len(a)
    return 2 * precision * recall / (precision + recall)


def _seq(a: List[str], b: List[str], sort: bool = False) -> float:
    if sort:
        a = sorted(a)
        b = sorted(b)
    return SequenceMatcher(None, " ".join(a), " ".join(b)).ratio()


def score_names(name1: str, name2: str) -> float:
    a = normalize_tokens(name1)
    b = normalize_tokens(name2)
    if not a or not b:
        return 0.0
    if a == b:
        return 1.0

    exact = _exact_f1(a, b)
    fuzzy = _fuzzy_f1(a, b)
    seq = _seq(a, b)
    sorted_seq = _seq(a, b, sort=True)

    score = 0.30 * exact + 0.30 * fuzzy + 0.20 * seq + 0.20 * sorted_seq

    # Reordered exact token set.
    if exact >= 0.99:
        score = max(score, 0.92)

    # Initial expansion is fine when at least one full token also matches.
    if exact > 0 and fuzzy >= 0.90 and len(a) >= 2 and len(b) >= 2:
        score = max(score, 0.78)

    # All-initial comparisons are inherently ambiguous.
    if all(_is_initial(t) for t in a) and all(_is_initial(t) for t in b):
        score *= 0.80

    # A lone leading initial for a first name is ambiguous ("R Kumar" could
    # be Rahul / Rajesh / Rakesh / Ravi...). Do not auto-MATCH on that
    # pattern alone. Trailing initials ("Rahul K") and multi-initial
    # patterns ("K S Rao") are unaffected.
    init_a = [i for i, t in enumerate(a) if _is_initial(t)]
    init_b = [i for i, t in enumerate(b) if _is_initial(t)]

    lone_lead_a = (init_a == [0] and not init_b)
    lone_lead_b = (init_b == [0] and not init_a)

    if (
        lone_lead_a and len(b[0]) >= 3 and b[0].startswith(a[0])
    ) or (
        lone_lead_b and len(a[0]) >= 3 and a[0].startswith(b[0])
    ):
        score = min(score, 0.65)

    return round(min(1.0, max(0.0, score)), 4)


def decide(score: float) -> str:
    if score >= MATCH_THRESHOLD:
        return "MATCH"
    if score >= REVIEW_THRESHOLD:
        return "REVIEW"
    return "NO_MATCH"


def explain(name1: str, name2: str) -> Dict[str, object]:
    a = normalize_tokens(name1)
    b = normalize_tokens(name2)
    score = score_names(name1, name2)
    return {
        "name1": name1,
        "name2": name2,
        "tokens1": a,
        "tokens2": b,
        "score": score,
        "decision": decide(score),
    }

Writing name_matcher/matcher.py


In [36]:
%%writefile name_matcher/evaluate.py
import csv
from typing import List, Tuple

from .matcher import score_names, decide


def _to_label(value: str) -> int:
    value = str(value).strip().lower()
    return 1 if value in {"1", "true", "yes", "match", "same"} else 0


def evaluate_csv(path: str) -> None:
    rows: List[Tuple[str, str, int, float, int, str]] = []

    with open(path, newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            y_true = _to_label(row["label"])
            score = score_names(row["name1"], row["name2"])
            decision = decide(score)
            y_pred = 1 if decision == "MATCH" else 0
            rows.append((row["name1"], row["name2"], y_true, score, y_pred, decision))

    tp = sum(1 for r in rows if r[2] == 1 and r[4] == 1)
    fp = sum(1 for r in rows if r[2] == 0 and r[4] == 1)
    tn = sum(1 for r in rows if r[2] == 0 and r[4] == 0)
    fn = sum(1 for r in rows if r[2] == 1 and r[4] == 0)

    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0
    accuracy = (tp + tn) / len(rows) if rows else 0.0

    print(f"n={len(rows)}")
    print(f"TP={tp} FP={fp} TN={tn} FN={fn}")
    print(f"precision={precision:.4f}")
    print(f"recall={recall:.4f}")
    print(f"f1={f1:.4f}")
    print(f"accuracy={accuracy:.4f}")
    print()
    print(f"{'name1':<22} {'name2':<22} {'label':>5} {'score':>7} {'decision':>10}")
    print("-" * 75)
    for n1, n2, y, s, _, d in rows:
        print(f"{n1:<22} {n2:<22} {y:>5} {s:>7.4f} {d:>10}")


if __name__ == "__main__":
    import argparse
    parser = argparse.ArgumentParser()
    parser.add_argument("csv", nargs="?", default="data/sample_pairs.csv")
    args = parser.parse_args()
    evaluate_csv(args.csv)

Writing name_matcher/evaluate.py


In [37]:
%%writefile data/sample_pairs.csv
name1,name2,label
Rahul Kumar,Rahul Kumar,1
Rahul Kumar,Rahul K,1
Rahul Kumar,K Rahul,1
Rajesh Kumar,Rajesh Kumr,1
Shri Rajesh Kumar,Rajesh Kumar,1
S/o Rajesh Kumar,Rajesh Kumar,1
Mohammed Irfan,Md Irfan,1
Mohd Irfan,Mohammed Irfan,1
K S Rao,Krishna Srinivas Rao,1
Krishna Rao,Krishna Srinivas Rao,1
Amit Kumar,Amit Kumar Singh,1
Amit Kumar,Amit K,1
Anjali Sharma,Anjali S,1
Anjali Sharma,Sharma Anjali,1
Sunita Devi,Smt Sunita Devi,1
Rakesh Kumar,Rakesh K,1
S. Kumar,Suresh Kumar,1
Suresh Kumar,Suresh K,1
Vijay Kumar,Vijaykumar,1
Rahul Kumar,Rahul Singh,0
Rajesh Kumar,Rakesh Kumar,0
Amit Kumar,Amit Singh,0
Anjali Sharma,Anjali Verma,0
Sunita Devi,Sunita Kumari,0
Rakesh Kumar,Rakesh Sharma,0
Suresh Kumar,Suresh Gupta,0
K S Rao,K R Rao,0
Krishna Rao,Krishna Reddy,0
Mohammed Irfan,Mohammed Imran,0
Amit Kumar,Sumit Kumar,0
Rajesh Kumar,Rajeshwari Kumar,0
Vijay Kumar,Vijay Singh,0
Anita Sharma,Anita Verma,0
Sunil Kumar,Sunil Gupta,0
Pooja Patel,Pooja Shah,0
Ravi Teja,Ravi Varma,0
Sneha Reddy,Sneha Rao,0
Vikas Agarwal,Vikas Agarwal,1
Manish Tiwari,Manish Tiwari,1
Ashish Pandey,Ashish Pandey,1
Nitin Joshi,Nitin Joshi,1
Pankaj Mishra,Pankaj Mishra,1
Saurabh Saxena,Saurabh Saxena,1
Naveen Nair,Naveen Nair,1
Sanjay Pillai,Sanjay Pillai,1
Rohit Chatterjee,Rohit Chatterjee,1
Varun Bansal,Varun Bansal,1
Siddharth Menon,Siddharth Menon,1
Abhishek Dubey,Abhishek Dubey,1
Rohan Iyer,Rohan Iyer,1
Kartik Bhatt,Kartik Bhatt,1
Aditya Choudhary,Aditya Choudhary,1
Prashant Hegde,Prashant Hegde,1
Ankur Trivedi,Ankur Trivedi,1
Gurpreet Singh,Gurpreet Singh,1
Harpreet Kaur,Harpreet Kaur,1
Jaspreet Singh,Jaspreet Singh,1
Vikas Agarwal,Vikas A,1
Manish Tiwari,Manish T,1
Ashish Pandey,Ashish P,1
Nitin Joshi,Nitin J,1
Pankaj Mishra,Pankaj M,1
Saurabh Saxena,Saurabh S,1
Naveen Nair,Naveen N,1
Sanjay Pillai,Sanjay P,1
Rohit Chatterjee,Rohit C,1
Varun Bansal,Varun B,1
Siddharth Menon,Siddharth M,1
Abhishek Dubey,Abhishek D,1
Rohan Iyer,Rohan I,1
Kartik Bhatt,Kartik B,1
Aditya Choudhary,Aditya C,1
Vikas Agarwal,Agarwal Vikas,1
Manish Tiwari,Tiwari Manish,1
Naveen Nair,Nair Naveen,1
Sanjay Pillai,Pillai Sanjay,1
Rohit Chatterjee,Chatterjee Rohit,1
Siddharth Menon,Menon Siddharth,1
Rohan Iyer,Iyer Rohan,1
Kartik Bhatt,Bhatt Kartik,1
Prashant Hegde,Hegde Prashant,1
Gurpreet Singh,Singh Gurpreet,1
Mr. Vikas Agarwal,Vikas Agarwal,1
Mrs. Harpreet Kaur,Harpreet Kaur,1
Dr. Manish Tiwari,Manish Tiwari,1
Shri Ramesh Deshpande,Ramesh Deshpande,1
Smt. Lakshmi Narayan,Lakshmi Narayan,1
Prof. Naveen Nair,Naveen Nair,1
Ms. Simran Kaur,Simran Kaur,1
Dr. Prashant Hegde,Prashant Hegde,1
Shri Suresh Iyengar,Suresh Iyengar,1
Smt. Saraswati Pillai,Saraswati Pillai,1
S/o Ramesh Deshpande,Ramesh Deshpande,1
D/o Lakshmi Narayan,Lakshmi Narayan,1
W/o Suresh Iyengar,Suresh Iyengar,1
Son of Vikram Singh,Vikram Singh,1
D/o Anil Kapoor,Anil Kapoor,1
W/o Rajan Mehta,Rajan Mehta,1
S/o Bhaskar Rao,Bhaskar Rao,1
D/o Kamala Devi,Kamala Devi,1
W/o Hari Prasad,Hari Prasad,1
S/o Madan Lal,Madan Lal,1
Abdul Rahman,Abdul Rehman,1
Md Salim,Mohammed Salim,1
Mohd Faizal,Mohammed Faizal,1
Md Asif,Mohammed Asif,1
Md Tariq,Mohammed Tariq,1
Mohammed Yasin,Md Yasin,1
Mohammed Karim,Mohd Karim,1
Md Rahim,Mohammed Rahim,1
Mohd Imran,Mohammed Imran,1
Md Imran,Mohammed Imran,1
Vikas Agarwal,Vikas Agrawal,1
Manish Tiwari,Maanish Tiwari,1
Ashish Pandey,Ashish Panday,1
Nitin Joshi,Nithin Joshi,1
Pankaj Mishra,Pankaj Misra,1
Saurabh Saxena,Saurabh Saksena,1
Sanjay Pillai,Sanjay Pillay,1
Rohit Chatterjee,Rohit Chatterji,1
Abhishek Dubey,Abhishek Dube,1
Rohan Iyer,Rohan Aiyer,1
Aditya Choudhary,Aditya Chaudhary,1
Prashant Hegde,Prashanth Hegde,1
Ankur Trivedi,Ankur Trivdi,1
Gurpreet Singh,Gurpreet Sing,1
Harpreet Kaur,Harpreet Kour,1
Arjun Kumar Mehta,Arjun Mehta,1
Ravi Shankar Rao,Ravi Rao,1
Sita Ram Sharma,Sita Sharma,1
Krishna Murthy Iyer,Krishna Iyer,1
Ram Prakash Singh,Ram Singh,1
Hari Om Sharma,Hari Sharma,1
Om Prakash Gupta,Om Gupta,1
Shiv Kumar Verma,Shiv Verma,1
Devendra Nath Mishra,Devendra Mishra,1
Ram Krishna Sharma,Ram Sharma,1
Jai Prakash Yadav,Jai Yadav,1
Satya Narayan Reddy,Satya Reddy,1
Ravi Chandran Pillai,Ravi Pillai,1
Bala Krishna Nair,Bala Nair,1
Adi Shankar Rao,Adi Rao,1
Vikas Agarwal,Vikas Sharma,0
Manish Tiwari,Manish Verma,0
Ashish Pandey,Ashish Mishra,0
Nitin Joshi,Nitin Desai,0
Pankaj Mishra,Pankaj Pandey,0
Saurabh Saxena,Saurabh Sinha,0
Naveen Nair,Naveen Menon,0
Sanjay Pillai,Sanjay Nair,0
Rohit Chatterjee,Rohit Banerjee,0
Varun Bansal,Varun Kapoor,0
Siddharth Menon,Siddharth Nair,0
Abhishek Dubey,Abhishek Tiwari,0
Rohan Iyer,Rohan Iyengar,0
Kartik Bhatt,Kartik Shah,0
Aditya Choudhary,Aditya Yadav,0
Prashant Hegde,Prashant Rao,0
Ankur Trivedi,Ankur Dubey,0
Gurpreet Singh,Gurpreet Kaur,0
Harpreet Kaur,Harpreet Singh,0
Jaspreet Singh,Jaspreet Kaur,0
Rahul Verma,Rahul Sharma,0
Rajesh Singh,Rajesh Kumar,0
Amit Sharma,Amit Verma,0
Anil Kapoor,Anil Kumar,0
Sunil Mehta,Sunil Sharma,0
Sanjay Gupta,Sanjay Verma,0
Manoj Kumar,Manoj Singh,0
Vijay Sharma,Vijay Verma,0
Ajay Patel,Ajay Shah,0
Suresh Nair,Suresh Menon,0
Ramesh Iyer,Ramesh Iyengar,0
Ravi Shankar,Ravi Prasad,0
Kiran Kumar,Kiran Reddy,0
Naresh Gupta,Naresh Jain,0
Mahesh Kumar,Mahesh Singh,0
Deepak Verma,Sanjay Verma,0
Manish Tiwari,Rakesh Tiwari,0
Ashish Pandey,Alok Pandey,0
Nitin Joshi,Vinit Joshi,0
Pankaj Mishra,Sanjay Mishra,0
Rohit Chatterjee,Amit Chatterjee,0
Varun Bansal,Arun Bansal,0
Kartik Bhatt,Nikhil Bhatt,0
Aditya Choudhary,Ankit Choudhary,0
Prashant Hegde,Prasad Hegde,0
Ankur Trivedi,Ankit Trivedi,0
Gurpreet Singh,Harjeet Singh,0
Harpreet Kaur,Harleen Kaur,0
Jaspreet Singh,Jaskaran Singh,0
Simran Kaur,Sukhman Kaur,0
Mohammed Salim,Mohammed Karim,0
Md Asif,Md Yasin,0
Mohammed Faizal,Mohammed Faisal,0
Mohammed Rahim,Mohammed Karim,0
Ayesha Khan,Ayesha Khatoon,0
Fatima Begum,Fatima Bi,0
Zara Sheikh,Zara Khan,0
Nida Ansari,Nida Sheikh,0
Imran Khan,Imran Ali,0
Pooja Iyer,Pooja Nair,0
Kavita Rao,Kavita Reddy,0
Divya Menon,Divya Nair,0
Priya Sharma,Priya Verma,0
Lakshmi Iyer,Lakshmi Iyengar,0
Deepa Nair,Deepa Menon,0
Sneha Pillai,Sneha Nair,0
Aarti Desai,Aarti Joshi,0
Neha Kapoor,Neha Khanna,0
Pooja Bansal,Pooja Bhatia,0

Writing data/sample_pairs.csv


In [38]:
%%writefile tests/test_matcher.py
from name_matcher.matcher import score_names, decide


def test_exact_match():
    assert score_names("Rahul Kumar", "Rahul Kumar") == 1.0


def test_initials_match():
    assert decide(score_names("Rahul Kumar", "Rahul K")) == "MATCH"


def test_title_and_relation_removed():
    assert score_names("Shri Rajesh Kumar", "Rajesh Kumar") == 1.0
    assert score_names("S/o Rajesh Kumar", "Rajesh Kumar") == 1.0


def test_alias():
    assert score_names("Md Irfan", "Mohammed Irfan") == 1.0


def test_reordered_name():
    assert decide(score_names("Rahul Kumar", "Kumar Rahul")) == "MATCH"


def test_different_surname_does_not_match():
    assert decide(score_names("Amit Kumar", "Amit Singh")) == "NO_MATCH"


def test_typo_matches():
    assert decide(score_names("Rajesh Kumar", "Rajesh Kumr")) == "MATCH"


def test_initial_only_is_conservative():
    assert decide(score_names("R Kumar", "Rajesh Kumar")) in {"REVIEW", "NO_MATCH"}

Writing tests/test_matcher.py


In [40]:
import os
print("CWD:", os.getcwd())
print("Files:", os.listdir("."))
print("Is name_matcher here?", os.path.isdir("name_matcher"))

CWD: /content/digio-name-match/digio-name-match/digio-name-match
Files: ['tests', 'data', 'NOTES.md', 'name_matcher']
Is name_matcher here? True


In [41]:
%%writefile pytest.ini
[pytest]
pythonpath = .
testpaths = tests

Writing pytest.ini


In [42]:
!pytest -q

........                                                                 [100%]
8 passed in 0.03s


In [43]:
from name_matcher import explain, score_names, decide

print(explain("Rahul Kumar", "Rahul K"))
print(explain("Mohammed Irfan", "Md Irfan"))
print(explain("Amit Kumar", "Amit Singh"))

print(score_names("Rajesh Kumar", "Rajesh Kumr"))
print(decide(score_names("Anjali Sharma", "Sharma Anjali")))

{'name1': 'Rahul Kumar', 'name2': 'Rahul K', 'tokens1': ['rahul', 'kumar'], 'tokens2': ['rahul', 'k'], 'score': 0.78, 'decision': 'MATCH'}
{'name1': 'Mohammed Irfan', 'name2': 'Md Irfan', 'tokens1': ['mohammed', 'irfan'], 'tokens2': ['mohammed', 'irfan'], 'score': 1.0, 'decision': 'MATCH'}
{'name1': 'Amit Kumar', 'name2': 'Amit Singh', 'tokens1': ['amit', 'kumar'], 'tokens2': ['amit', 'singh'], 'score': 0.5, 'decision': 'NO_MATCH'}
0.8326
MATCH


In [44]:
!python -m name_matcher "Rahul Kumar" "Rahul K"

{
  "name1": "Rahul Kumar",
  "name2": "Rahul K",
  "tokens1": [
    "rahul",
    "kumar"
  ],
  "tokens2": [
    "rahul",
    "k"
  ],
  "score": 0.78,
  "decision": "MATCH"
}


In [45]:
!python -m name_matcher.evaluate data/sample_pairs.csv

n=211
TP=116 FP=2 TN=85 FN=8
precision=0.9831
recall=0.9355
f1=0.9587
accuracy=0.9526

name1                  name2                  label   score   decision
---------------------------------------------------------------------------
Rahul Kumar            Rahul Kumar                1  1.0000      MATCH
Rahul Kumar            Rahul K                    1  0.7800      MATCH
Rahul Kumar            K Rahul                    1  0.7800      MATCH
Rajesh Kumar           Rajesh Kumr                1  0.8326      MATCH
Shri Rajesh Kumar      Rajesh Kumar               1  1.0000      MATCH
S/o Rajesh Kumar       Rajesh Kumar               1  1.0000      MATCH
Mohammed Irfan         Md Irfan                   1  1.0000      MATCH
Mohd Irfan             Mohammed Irfan             1  1.0000      MATCH
K S Rao                Krishna Srinivas Rao       1  0.7800      MATCH
Krishna Rao            Krishna Srinivas Rao       1  0.7639      MATCH
Amit Kumar             Amit Kumar Singh           1  0.7